In [16]:
import pandas as pd
import numpy as np

In [17]:
docs = pd.DataFrame({
    'text': ['quick is brown fox', 'fox brown fox', 'lazy dog quick', 'fox quick dog' ],
    'label': [1, 0, 0, 1]
})

In [18]:
docs

,text,label
0,quick is brown fox,1
1,fox brown fox,0
2,lazy dog quick,0
3,fox quick dog,1


### Bag of Words

Represents each document as a vector of token counts (or presence), using a fixed vocabulary — order and grammar are ignored; only token frequency matters.

Pros
- Very simple and fast to implement.
- Interpretable: each feature maps to a specific word.

Cons
- Loses word order and context (cannot capture meaning or syntax).
- Vocabulary can become very large → high dimensionality and sparsity.
- Sensitive to tokenization, stop words, and rare words.
- Cannot model semantics, polysemy, or word relationships (use embeddings or n‑grams/contextual models to address).

In [ ]:
from sklearn.feature_extraction.text import CountVectorizer

cv_bow = CountVectorizer() # for OHE, just set binary=True 
vectorized_docs_bow = cv_bow.fit_transform(docs['text'])
cv_bow.vocabulary_ 

{'quick': 5, 'is': 3, 'brown': 0, 'fox': 2, 'lazy': 4, 'dog': 1}

In [41]:
pd.DataFrame(vectorized_docs_bow.toarray(), columns=cv_bow.get_feature_names_out()).join(docs['text'])

,brown,dog,fox,is,lazy,quick,text
0,1,0,1,1,0,1,quick is brown fox
1,1,0,2,0,0,0,fox brown fox
2,0,1,0,0,1,1,lazy dog quick
3,0,1,1,0,0,1,fox quick dog


### N-grams

Represents documents as sequences of n consecutive tokens (words). Unlike Bag of Words, n-grams capture local word order and context by grouping adjacent tokens.

**Examples:**
- Bigrams (n=2): "quick brown", "brown fox"
- Trigrams (n=3): "quick brown fox"

**Pros**
- Captures word order and local context, improving semantic understanding, so better performance than Bag of Words for many NLP tasks.
- Can model common phrases and collocations (e.g., "New York").

- Simple to implement and interpretable.

**Cons**
- Dimensionality increases exponentially with n (curse of dimensionality).  
    - Better: limit n, restrict vocabulary, apply feature selection, or use hashing (FeatureHasher) / embeddings.
- Sparse representations for larger n values.  
    - Better: use sparse-aware models, feature hashing, or apply dimensionality reduction (TruncatedSVD).
- Still ignores long-range dependencies and syntactic structure.  
    - Better: combine with embeddings or use sequence models (RNNs/Transformers) and syntactic features (POS, dependency).
- Requires more data and computational resources than unigrams.  
    - Better: use pretrained embeddings, regularization, subsampling, or lower n; leverage efficient vectorizers.
- May not generalize well to unseen n-grams.  
    - Better: use character/subword n-grams, smoothing/backoff techniques, or contextual embeddings (BERT, fastText).

In [21]:
from sklearn.feature_extraction.text import CountVectorizer

# CountVectorizer with ngram_range of (2, 2) for bigrams
cv_ngram = CountVectorizer(ngram_range=(2, 2))
vectorized_docs_ngram = cv_ngram.fit_transform(docs['text'])
cv_ngram.vocabulary_

{'quick is': 7,
 'is brown': 4,
 'brown fox': 0,
 'fox brown': 2,
 'lazy dog': 5,
 'dog quick': 1,
 'fox quick': 3,
 'quick dog': 6}

In [22]:
pd.DataFrame(vectorized_docs_ngram.toarray(), columns=cv_ngram.get_feature_names_out()).join(docs['text'])

,brown fox,dog quick,fox brown,fox quick,is brown,lazy dog,quick dog,quick is,text
0,1,0,0,0,1,0,0,1,quick is brown fox
1,1,0,1,0,0,0,0,0,fox brown fox
2,0,1,0,0,0,1,0,0,lazy dog quick
3,0,0,0,1,0,0,1,0,fox quick dog


### Tf-Idf Vectorization

Tf‑Idf (term frequency–inverse document frequency) scores each term by how often it appears in a document (TF) scaled by how rare it is across the corpus (IDF). Produces sparse numeric vectors that emphasize discriminative words and downweight common tokens.

Pros
- Highlights important/diagnostic terms.
- Simple and fast to compute; works with sparse linear models.
- Interpretable: feature = specific token or n‑gram.
- Language-agnostic; easy to combine with classic ML pipelines.

Cons (and better alternatives)
- Ignores word order and phrase structure. 
    - Better: use n‑grams or sequence models (RNNs/Transformers).
- Cannot capture polysemy (multiple meaning of same word e.g. bank) or context-dependent meaning.  
    - Better: contextual embeddings (BERT, RoBERTa).
- High dimensionality and sparsity for large vocabularies. 
    - Better: feature hashing, vocabulary pruning, or dimensionality reduction (TruncatedSVD)..


In [32]:
from sklearn.feature_extraction.text import TfidfVectorizer

tfidf_vectorizer = TfidfVectorizer(ngram_range=(1, 2))
vectorized_docs_tfidf = tfidf_vectorizer.fit_transform(docs['text'])

In [ ]:
# values are ranks not counts
tfidf_vectorizer.vocabulary_

{'quick': 11,
 'is': 7,
 'brown': 0,
 'fox': 4,
 'quick is': 13,
 'is brown': 8,
 'brown fox': 1,
 'fox brown': 5,
 'lazy': 9,
 'dog': 2,
 'lazy dog': 10,
 'dog quick': 3,
 'fox quick': 6,
 'quick dog': 12}

In [33]:
pd.DataFrame(vectorized_docs_tfidf.toarray(), columns=tfidf_vectorizer.get_feature_names_out()).join(docs['text'])

,brown,brown fox,dog,dog quick,fox,fox brown,fox quick,is,is brown,lazy,lazy dog,quick,quick dog,quick is,text
0,0.350561,0.350561,0.000000,0.000000,0.283809,0.000000,0.000000,0.444642,0.444642,0.000000,0.000000,0.283809,0.000000,0.444642,quick is brown fox
1,0.400626,0.400626,0.000000,0.000000,0.648682,0.508143,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,fox brown fox
2,0.000000,0.000000,0.392784,0.498197,0.000000,0.000000,0.000000,0.000000,0.000000,0.498197,0.498197,0.317993,0.000000,0.000000,lazy dog quick
3,0.000000,0.000000,0.425305,0.000000,0.344321,0.000000,0.539445,0.000000,0.000000,0.000000,0.000000,0.344321,0.539445,0.000000,fox quick dog
